In [1]:
import torch

Simple Self-Attention Without Trainable Weights

In [2]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],
        [0.55, 0.87, 0.66],
        [0.57, 0.85, 0.64],
        [0.22, 0.58, 0.33],
        [0.77, 0.25, 0.10],
        [0.05, 0.80, 0.55]
    ]
)

In [3]:
input_2 = inputs[1]

attention_scores = torch.empty(inputs.shape[0])

for idx, ele in enumerate(inputs):        
    attention_scores[idx] = torch.dot(input_2, inputs[idx])


attention_scores

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

In [4]:
attention_weights = torch.softmax(attention_scores,dim=0) #normalize weights
attention_weights


tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [5]:
context_vec = torch.zeros(input_2.shape[0])
for idx, ele in enumerate(inputs):
    context_vec += attention_weights[idx] * inputs[idx]

context_vec


tensor([0.4419, 0.6515, 0.5683])

In [6]:
attention_score_matrix = torch.empty(inputs.shape[0], inputs.shape[0])

for query_i in range(0, inputs.shape[0]):
    for idx in range(0, inputs.shape[0]):
        attention_score_matrix[query_i][idx] = torch.dot(inputs[query_i] , inputs[idx])

attention_score_matrix


tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [7]:
attention_weight_matrix = torch.softmax(attention_score_matrix, dim=1)
attention_weight_matrix

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [8]:
context_matrix = torch.zeros(inputs.shape[0], inputs[0].shape[0])

for query_i in range(0, inputs.shape[0]):
    for idx in range(0, inputs.shape[0]):
        context_matrix[query_i] += attention_weight_matrix[query_i][idx] * inputs[idx]

context_matrix


tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [9]:
#using matrix multiplication
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim = 1)
context_vecs = attn_weights @ inputs

context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

Self-Attention with Trainable Weights

In [24]:
#testing for single query
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
W_keys = torch.nn.Parameter(torch.rand(d_in, d_out))
W_values = torch.nn.Parameter(torch.rand(d_in, d_out))

query_2 = x_2 @ W_query
keys = inputs @ W_keys
values = inputs @ W_values

attn_scores_2 = query_2 @ keys.T

d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2/d_k**0.5, dim = -1)
attn_weights_2

context_vecs_2 = attn_weights_2 @ values
context_vecs_2

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

In [29]:
class SelfAttention(torch.nn.Module):

    def __init__(self, d_in, d_out, qkv_bias = False):  
        super().__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_keys = torch.nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_values = torch.nn.Linear(d_in, d_out, bias = qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_keys(x)
        values = self.W_values(x)

        attn_scores = queries @ keys.T
        
        d_k = keys.shape[1]
        attn_weights = torch.softmax(attn_scores/d_k**0.5, dim = -1)
        context_vec = attn_weights @ values

        return context_vec

In [30]:

d_in = inputs.shape[1]
d_out = 2

torch.manual_seed(789)

model = SelfAttention(d_in, d_out)
model(inputs)


tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)